# ',torch.cuda.is_available(),torch.cuda.device_count()); print('HF_CACHE',os.path.exists('/kaggle/working/hf_cache'))

In [ ]:
import os, torch, shutil
print('DIAG_START')
print('CUDA', torch.cuda.is_available(), 'GPU_COUNT', torch.cuda.device_count())
print('GPUS', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('DISK_FREE_GB', round(shutil.disk_usage('/kaggle/working').free/1024**3,2))
print('HF_HOME', os.environ.get('HF_HOME'))
print('WORKING_FILES', os.listdir('/kaggle/working')[:25])
print('HF_CACHE_EXISTS', os.path.exists(os.path.expanduser('~/.cache/huggingface')))
print('DIAG_END')

In [ ]:
import os, torch, shutil, glob, diffusers, transformers, huggingface_hub
print('VERSIONS', torch.__version__, diffusers.__version__, transformers.__version__, huggingface_hub.__version__)
print('GPU_FREE_GB', [round(torch.cuda.mem_get_info(i)[0]/1024**3,2) for i in range(torch.cuda.device_count())])
for p in ['/kaggle/working/hf_cache','/root/.cache/huggingface','/kaggle/working']:
     print('DIR',p,os.path.exists(p))
     if os.path.isdir(p): print('CONTENTS',[(os.path.basename(x),round(os.path.getsize(x)/1024**2,1) if os.path.isfile(x) else 'DIR') for x in glob.glob(p+'/*')[:30]])

In [ ]:
print('BASE_PIPELINE_LOAD_BEGIN')
print('test')

In [ ]:
print('FRESH_CELL_READY')

In [ ]:
import os, torch, time, traceback
from diffusers import StableDiffusionPipeline
base='/kaggle/working/hf_cache/models--Yntec--AnythingV5/snapshots/ca6595b27b8dce279b8f2b075a1e373907fda95b'
adapter='/kaggle/working/hf_cache/models--h94--IP-Adapter/snapshots/018e402774aeeddd60609b4ecdb7e298259dc729'
try:
        t0=time.time()
        pipe=StableDiffusionPipeline.from_pretrained(base, torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False, local_files_only=True)
        pipe=pipe.to('cuda')
        pipe.enable_attention_slicing()
        pipe.enable_vae_slicing()
        print('BASE_PIPELINE_READY',round(time.time()-t0,1),'sec')
        pipe.load_ip_adapter(adapter, subfolder='models', weight_name='ip-adapter_sd15_light.safetensors', image_encoder_folder='image_encoder')
        pipe.set_ip_adapter_scale(0.6)
        print('IP_ADAPTER_READY',round(time.time()-t0,1),'sec')
        print('GPU_FREE_GB',[round(torch.cuda.mem_get_info(i)[0]/1024**3,2) for i in range(torch.cuda.device_count())])
except Exception as e:
        print('LOAD_FAILED',type(e).__name__,str(e)[:1500])
        traceback.print_exc(limit=3)

In [ ]:
import os, torch, time, traceback
from diffusers import StableDiffusionPipeline
base='/kaggle/working/hf_cache/models--Yntec--AnythingV5/snapshots/ca6595b27b8dce279b8f2b075a1e373907fda95b'
adapter='/kaggle/working/hf_cache/models--h94--IP-Adapter/snapshots/018e402774aeeddd60609b4ecdb7e298259dc729'
try:
        t0=time.time()
        pipe=StableDiffusionPipeline.from_pretrained(base, torch_dtype=torch.float16, variant='fp16', safety_checker=None, requires_safety_checker=False, local_files_only=True)
        pipe=pipe.to('cuda')
        pipe.enable_attention_slicing()
        pipe.enable_vae_slicing()
        print('BASE_PIPELINE_READY',round(time.time()-t0,1),'sec')
        pipe.load_ip_adapter(adapter, subfolder='models', weight_name='ip-adapter_sd15_light.safetensors', image_encoder_folder='image_encoder')
        pipe.set_ip_adapter_scale(0.6)
        print('IP_ADAPTER_READY',round(time.time()-t0,1),'sec')
        print('GPU_FREE_GB',[round(torch.cuda.mem_get_info(i)[0]/1024**3,2) for i in range(torch.cuda.device_count())])
except Exception as e:
        print('LOAD_FAILED',type(e).__name__,str(e)[:1500])
        traceback.print_exc(limit=3)

In [ ]:
import time,traceback
try:
        pipe.disable_attention_slicing()
        t0=time.time()
        pipe.load_ip_adapter(adapter, subfolder='models', weight_name='ip-adapter_sd15_light.safetensors', image_encoder_folder='image_encoder')
        pipe.set_ip_adapter_scale(0.6)
        print('IP_ADAPTER_READY',round(time.time()-t0,1),'sec')
        print('GPU_FREE_GB',[round(torch.cuda.mem_get_info(i)[0]/1024**3,2) for i in range(torch.cuda.device_count())])
except Exception as e:
        print('IP_ADAPTER_FAILED',type(e).__name__,str(e)[:1800])
        traceback.print_exc(limit=4)

In [ ]:
from PIL import Image, ImageOps, ImageDraw
import torch, time, os, traceback
ref_path='/kaggle/working/heroine_reference_seed20261123.png'
ref=Image.open(ref_path).convert('RGB')
print('REFERENCE',ref.size,os.path.getsize(ref_path))
prompts=[
     'same young adult anime fantasy heroine as the reference image, long glossy black hair, violet eyes, delicate face, wearing the same dark navy and silver fantasy outfit, front view, neutral standing pose, clean light background, full character design sheet',
     'same young adult anime fantasy heroine as the reference image, long glossy black hair, violet eyes, delicate face, wearing the same dark navy and silver fantasy outfit, strict side profile facing right, neutral standing pose, clean light background, character design sheet',
     'same young adult anime fantasy heroine as the reference image, long glossy black hair, violet eyes, delicate face, wearing the same dark navy and silver fantasy outfit, full body view, standing pose, clean light background, character design sheet'
]
files=[]
for i,prompt in enumerate(prompts):
        t0=time.time()
        g=torch.Generator(device='cuda').manual_seed(20261124+i)
        result=pipe(prompt, ip_adapter_image=ref, num_inference_steps=18, guidance_scale=6.5, height=512, width=512, generator=g)
        out=f'/kaggle/working/ipadapter_view_{i+1}.png'
        result.images[0].save(out)
        files.append(out)
        print('VIEW_SAVED',i+1,out,round(time.time()-t0,1),'sec')
    thumbs=[Image.open(p).convert('RGB').resize((384,384)) for p in files]
sheet=Image.new('RGB',(384*3,420),(245,245,245))
for i,im in enumerate(thumbs):
        sheet.paste(im,(384*i,0))
        ImageDraw.Draw(sheet).text((384*i+12,390),['FRONT','SIDE','FULL BODY'][i],fill=(20,20,20))
    sheet.save('/kaggle/working/ipadapter_consistency_contact_sheet.png')
print('CONTACT_SHEET_SAVED','/kaggle/working/ipadapter_consistency_contact_sheet.png')
]

In [ ]:
from PIL import Image, ImageDraw
import torch, time, os
ref=Image.open('/kaggle/working/heroine_reference_seed20261123.png').convert('RGB')
print('REFERENCE',ref.size)
t0=time.time()
g=torch.Generator(device='cuda').manual_seed(20261124)
im1=pipe('same young adult anime fantasy heroine as reference, long glossy black hair, violet eyes, delicate face, same dark navy and silver outfit, front view, neutral standing pose, clean light background, character design sheet',ip_adapter_image=ref,num_inference_steps=16,guidance_scale=6.5,height=512,width=512,generator=g).images[0]
im1.save('/kaggle/working/ipadapter_view_front.png')
print('FRONT_SAVED')
g=torch.Generator(device='cuda').manual_seed(20261125)
im2=pipe('same young adult anime fantasy heroine as reference, long glossy black hair, violet eyes, delicate face, same dark navy and silver outfit, strict right-facing side profile, neutral standing pose, clean light background, character design sheet',ip_adapter_image=ref,num_inference_steps=16,guidance_scale=6.5,height=512,width=512,generator=g).images[0]
im2.save('/kaggle/working/ipadapter_view_side.png')
print('SIDE_SAVED')
g=torch.Generator(device='cuda').manual_seed(20261126)
im3=pipe('same young adult anime fantasy heroine as reference, long glossy black hair, violet eyes, delicate face, same dark navy and silver outfit, full body view, standing pose, clean light background, character design sheet',ip_adapter_image=ref,num_inference_steps=16,guidance_scale=6.5,height=512,width=512,generator=g).images[0]
im3.save('/kaggle/working/ipadapter_view_full.png')
print('FULL_SAVED','SECONDS',round(time.time()-t0,1))
sheet=Image.new('RGB',(1536,540),'white')
sheet.paste(im1.resize((512,512)),(0,0)); sheet.paste(im2.resize((512,512)),(512,0)); sheet.paste(im3.resize((512,512)),(1024,0))
d=ImageDraw.Draw(sheet); d.text((15,515),'FRONT',fill='black'); d.text((527,515),'SIDE',fill='black'); d.text((1039,515),'FULL BODY',fill='black')
sheet.save('/kaggle/working/ipadapter_consistency_contact_sheet.png')
print('CONTACT_SHEET_SAVED','/kaggle/working/ipadapter_consistency_contact_sheet.png')from PIL import Image, ImageDraw
import torch,time
ref=Image.open('/kaggle/working/heroine_reference_seed20261123.png').convert('RGB')
pipe.set_ip_adapter_scale(0.82)
neg='multiple people, duplicate person, twins, two characters, extra head, extra body, collage, split screen, text, watermark, blurry, low quality, deformed hands'
g=torch.Generator(device='cuda').manual_seed(20261127)
a=pipe('solo, one young adult anime fantasy heroine, same identity as reference, long glossy black hair, violet eyes, pale face, dark navy and silver fantasy outfit, front-facing portrait, centered single character, plain light gray background',ip_adapter_image=ref,negative_prompt=neg,num_inference_steps=18,guidance_scale=6.5,height=512,width=512,generator=g).images[0]
a.save('/kaggle/working/ipadapter_v2_front.png')
print('V2_FRONT_SAVED')
g=torch.Generator(device='cuda').manual_seed(20261128)
b=pipe('solo, one young adult anime fantasy heroine, same identity as reference, long glossy black hair, violet eyes, pale face, dark navy and silver fantasy outfit, strict side profile facing right, centered single character, plain light gray background',ip_adapter_image=ref,negative_prompt=neg,num_inference_steps=18,guidance_scale=6.5,height=512,width=512,generator=g).images[0]
b.save('/kaggle/working/ipadapter_v2_side.png')
print('V2_SIDE_SAVED')
g=torch.Generator(device='cuda').manual_seed(20261129)
c=pipe('solo, one young adult anime fantasy heroine, same identity as reference, long glossy black hair, violet eyes, pale face, dark navy and silver fantasy outfit, full body head to toe, standing alone, centered single character, plain light gray background',ip_adapter_image=ref,negative_prompt=neg,num_inference_steps=18,guidance_scale=6.5,height=512,width=512,generator=g).images[0]
c.save('/kaggle/working/ipadapter_v2_full.png')
print('V2_FULL_SAVED')
sheet=Image.new('RGB',(1536,540),'white');sheet.paste(a,(0,0));sheet.paste(b,(512,0));sheet.paste(c,(1024,0));d=ImageDraw.Draw(sheet);d.text((15,515),'FRONT',fill='black');d.text((527,515),'SIDE',fill='black');d.text((1039,515),'FULL BODY',fill='black');sheet.save('/kaggle/working/ipadapter_v2_contact_sheet.png');print('V2_CONTACT_SAVED','TIME',round(time.time(),1))

In [ ]:
from PIL import Image
from IPython.display import display
display(Image.open('/kaggle/working/ipadapter_consistency_contact_sheet.png'))

# 已验证的 IP-Adapter 人物参考图流程（2026-11-23）

本 Notebook 的实验环境已在 Kaggle 免费 T4 x2 会话中验证：CUDA 可用，检测到两张 Tesla T4。工作目录中已有 AnythingV5 SD1.5 Diffusers 模型和 `h94/IP-Adapter` 的 `ip-adapter_sd15_light.safetensors` 与 image encoder 缓存。

## 已验证步骤
- AnythingV5 以 `torch_dtype=torch.float16, variant="fp16"` 从本地缓存加载成功。
- `pipe.disable_attention_slicing()` 后，`pipe.load_ip_adapter(..., subfolder="models", weight_name="ip-adapter_sd15_light.safetensors", image_encoder_folder="image_encoder")` 加载成功。
- 使用同一张 `heroine_reference_seed20261123.png` 作为 `ip_adapter_image`，成功生成正面、侧面、全身三个 512×512 测试视图和 `ipadapter_consistency_contact_sheet.png`。

## 重要兼容性记录
- 缓存权重是 fp16 变体，必须指定 `variant="fp16"`，否则 Diffusers 会寻找不存在的默认权重文件。
- 不要在加载 IP-Adapter 之前启用 `enable_attention_slicing()`；本次会触发 `SlicedAttnProcessor` 构造错误。先加载 IP-Adapter，再按需测试内存优化选项。

## 结果边界
本次证明了模型、适配器和参考图控制链路可运行，不代表已经达到稳定的角色一致性。初次对照图仍存在视角构图偏差和重复人物倾向；需要继续调整提示词、适配器权重，并做多轮对照评估。当前结果仅为 512×512 原型测试，不是 1080P 成片。
